# Benford Law Notebook

This notebook follows the Benford workflow step by step.
The goal is simple:

- load the VynFi journal entry data
- check the first digit pattern
- score each general ledger account on the train side
- use those train side flags on the test side
- measure precision, recall, and F1


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold


In [2]:
from pathlib import Path

# This makes the notebook work whether you open it from the repo root
# or from inside the notebooks folder.
if (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    REPO_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not find the repo root")

DATA_DIR = REPO_ROOT / "data" / "training" / "vynfi"
OUT_DIR = REPO_ROOT / "data" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Data folder:", DATA_DIR)


Repo root: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint
Data folder: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint\data\training\vynfi


In [3]:
# The dataset is stored in three parquet files, so I load each part and join them.
shard_names = [
    "train-00000-of-00003.parquet",
    "train-00001-of-00003.parquet",
    "train-00002-of-00003.parquet",
]

frames = []
for name in shard_names:
    file_path = DATA_DIR / name
    frame = pd.read_parquet(file_path)
    frames.append(frame)
    print(name, frame.shape)

main_data = pd.concat(frames, ignore_index=True)
print("Full data shape:", main_data.shape)
main_data.head()


train-00000-of-00003.parquet (200000, 48)


train-00001-of-00003.parquet (176959, 48)


train-00002-of-00003.parquet (290625, 48)
Full data shape: (667584, 48)


,document_id,company_code,fiscal_year,fiscal_period,posting_date,document_date,document_type,currency,exchange_rate,reference,...,tax_code,transaction_id,account_class,account_class_name,account_sub_class,account_sub_class_name,predecessor_line_id,trading_partner,fraud_type,anomaly_type
0,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,d35dc4bc-97c1-5ed5-943c-c309f2fc0910,A.A,Cash & Cash Equivalents,A.A.A,Operating Cash,None,None,None,LatePosting
1,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,35c015fd-d1ff-5249-b762-f358654e2380,A.D,Prepaid Expenses & Other Current Assets,A.D.A,Prepaid Expenses,None,None,None,LatePosting
2,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,dad767d0-5c32-5eb4-9307-15d880670d07,A.H,Other Long-term Assets,A.H.A,Other Assets,None,None,None,LatePosting
3,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,a45c4cbd-98a0-5303-9a2e-752cd56bebb1,A.B,Trade Receivables,A.B.A,Trade Accounts Receivable,None,None,None,LatePosting
4,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,b1b232bc-c7bf-5d91-9ead-714e740ec1eb,A.C,Inventory,A.C.A,Inventory,None,None,None,LatePosting


In [4]:
empty_columns = [
    "auxiliary_account_number",
    "auxiliary_account_label",
    "lettrage",
    "lettrage_date",
    "tax_code",
]

# These columns directly describe anomalies, so keeping them would leak the answer.
leakage_columns = ["fraud_type", "anomaly_type", "is_anomaly"]

work_df = main_data.drop(columns=empty_columns, errors="ignore")

# I group by document_id so lines from the same journal entry cannot appear on both sides.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_index, test_index = next(
    splitter.split(work_df, y=work_df["is_fraud"], groups=work_df["document_id"])
)

train_df = work_df.iloc[train_index].reset_index(drop=True)
test_df = work_df.iloc[test_index].reset_index(drop=True)

y_test = test_df["is_fraud"].astype(int)

train_df = train_df.drop(columns=leakage_columns, errors="ignore")
test_df = test_df.drop(columns=leakage_columns, errors="ignore")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train fraud rate:", train_df["is_fraud"].mean())
print("Test fraud rate:", test_df["is_fraud"].mean())
print("Shared document ids:", len(set(train_df["document_id"]) & set(test_df["document_id"])))


Train shape: (536395, 40)
Test shape: (131189, 40)
Train fraud rate: 0.062409232002535446
Test fraud rate: 0.06194879143830657
Shared document ids: 0


In [5]:
expected_digits = np.arange(1, 10)
expected_shares = np.log10(1 + 1 / expected_digits)
# Skip amounts below one cent, use the standard 0.015 MAD limit, and require 500 rows to reduce random variation.
min_amount = 0.01
mad_threshold = 0.015
min_rows = 500

# Debit and credit are separate columns, so I combine them into one positive amount.
def line_amounts(df):
    debit = df["debit_amount"].fillna(0)
    credit = df["credit_amount"].fillna(0)
    return (debit + credit).abs()


# Benford only needs the first non-zero digit from each usable amount.
def first_digits(amount_series):
    usable = amount_series[amount_series >= min_amount]
    scaled = usable / np.power(10.0, np.floor(np.log10(usable)))
    return scaled.astype(int).clip(1, 9)


# I convert the digit counts into proportions so they can be compared with Benford.
def digit_shares(digit_series):
    counts = digit_series.value_counts().reindex(expected_digits, fill_value=0).to_numpy(dtype=float)
    total = counts.sum()
    if total == 0:
        return np.zeros(9)
    return counts / total


# MAD gives one simple score for the average gap from the expected proportions.
def mad_score(observed):
    return float(np.mean(np.abs(observed - expected_shares)))


In [6]:
# This table helps me see which first digits differ most from the expected pattern.
all_digits = first_digits(line_amounts(test_df))
observed = digit_shares(all_digits)

digit_table = pd.DataFrame(
    {
        "digit": expected_digits,
        "expected": expected_shares,
        "observed": observed,
        "difference": observed - expected_shares,
    }
)

digit_table


,digit,expected,observed,difference
0,1,0.301030,0.302599,0.001569
1,2,0.176091,0.173446,-0.002645
2,3,0.124939,0.121681,-0.003258
3,4,0.096910,0.096615,-0.000295
4,5,0.079181,0.081922,0.002741
5,6,0.066947,0.068557,0.001610
6,7,0.057992,0.057574,-0.000418
7,8,0.051153,0.051231,0.000078
8,9,0.045757,0.046376,0.000619


In [7]:
# I decide which accounts fail using only training data, then check those accounts on test data.
train_amounts = line_amounts(train_df)
train_digits = first_digits(train_amounts)

train_benford_data = pd.DataFrame(
    {
        "gl_account": train_df.loc[train_digits.index, "gl_account"].to_numpy(),
        "digit": train_digits.to_numpy(),
    }
)

rows = []
for gl_account, chunk in train_benford_data.groupby("gl_account"):
    observed = digit_shares(chunk["digit"])
    mad_value = mad_score(observed)
    rows.append(
        {
            "gl_account": gl_account,
            "rows": len(chunk),
            "mad": mad_value,
            "testable": len(chunk) >= min_rows,
            "flagged": len(chunk) >= min_rows and mad_value > mad_threshold,
        }
    )

group_scores = pd.DataFrame(rows).sort_values("mad", ascending=False).reset_index(drop=True)
failing_accounts = set(group_scores.loc[group_scores["flagged"], "gl_account"])

test_pred = test_df["gl_account"].isin(failing_accounts).astype(int)

# I report precision, recall and F1 because fraud data is imbalanced and accuracy can mislead.
precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)

print("Flagged accounts:", len(failing_accounts))
print("Flagged rows in test set:", int(test_pred.sum()))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))

group_scores.head(15)


Flagged accounts: 62
Flagged rows in test set: 13035
Precision: 0.0576
Recall: 0.0924
F1: 0.071


,gl_account,rows,mad,testable,flagged
0,4810,2,0.183091,False,False
1,4800,2,0.183091,False,False
2,500560,542,0.025913,True,True
3,500120,974,0.025578,True,True
4,500460,324,0.024682,False,False
5,500490,846,0.022306,True,True
6,200650,617,0.021871,True,True
7,6950,478,0.021565,False,False
8,100770,508,0.021368,True,True
9,500850,960,0.021237,True,True


In [8]:
# I save the tables so the results can be reused without running the notebook again.
digit_table.to_csv(OUT_DIR / "benford_digit_table.csv", index=False)
group_scores.to_csv(OUT_DIR / "benford_groups.csv", index=False)

benford_summary = pd.DataFrame(
    [
        {
            "method": "Benford (MAD by gl_account)",
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
            "n_flagged": int(test_pred.sum()),
        }
    ]
)

benford_summary


,method,precision,recall,f1,n_flagged
0,Benford (MAD by gl_account),0.0576,0.0924,0.071,13035
